In [11]:
import pandas as pd
import numpy as np

# ✅ Correct paths based on your folder structure
# ✅ Correct paths (based on your screenshot)

DATA_PATH = r"D:\ML_Project\emotion-drift-project\data\eyet4empathy\processed\merged_eye.csv"
OUTPUT_PATH = r"D:\ML_Project\emotion-drift-project\data\eyet4empathy\processed\eye_clean.csv"

CHUNK_SIZE = 100000

In [12]:
import pandas as pd
import numpy as np

FINAL_COLUMNS = [
    'Recording timestamp', 'Participant name', 'Recording name',
    'Gaze point X', 'Gaze point Y',
    'Gaze point left X', 'Gaze point left Y',
    'Gaze point right X', 'Gaze point right Y',
    'Gaze direction left X', 'Gaze direction left Y', 'Gaze direction left Z',
    'Gaze direction right X', 'Gaze direction right Y', 'Gaze direction right Z',
    'Pupil diameter left', 'Pupil diameter right',
    'Eye position left X (DACSmm)', 'Eye position left Y (DACSmm)', 'Eye position left Z (DACSmm)',
    'Eye position right X (DACSmm)', 'Eye position right Y (DACSmm)', 'Eye position right Z (DACSmm)',
    'Gaze point left X (DACSmm)', 'Gaze point left Y (DACSmm)',
    'Gaze point right X (DACSmm)', 'Gaze point right Y (DACSmm)',
    'Gaze point X (MCSnorm)', 'Gaze point Y (MCSnorm)',
    'Gaze point left X (MCSnorm)', 'Gaze point left Y (MCSnorm)',
    'Gaze point right X (MCSnorm)', 'Gaze point right Y (MCSnorm)',
    'Gaze event duration',
    'Fixation point X', 'Fixation point Y',
    'Fixation point X (MCSnorm)', 'Fixation point Y (MCSnorm)'
]

In [13]:
def clean_deep_learning_data(df):

    df = df.sort_values(['Participant name', 'Recording timestamp'])

    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

    important_numeric = [
        'Gaze point X', 'Gaze point Y',
        'Gaze point left X', 'Gaze point left Y',
        'Gaze point right X', 'Gaze point right Y'
    ]

    valid_numeric = [col for col in important_numeric if col in numeric_cols]

    if valid_numeric:
        df[valid_numeric] = df.groupby('Participant name')[valid_numeric].transform(
            lambda x: x.interpolate(method='linear', limit=5)
        )

        df[valid_numeric] = df.groupby('Participant name')[valid_numeric].transform(
            lambda x: x.ffill().bfill()
        )

    # categorical safe fill
    cat_to_fill = [c for c in categorical_cols if c not in ['Participant name']]

    if cat_to_fill:
        df[cat_to_fill] = df.groupby('Participant name')[cat_to_fill].transform(
            lambda x: x.ffill().bfill()
        )

    return df

In [14]:
print("🚀 Starting cleaning...")

chunks = pd.read_csv(
    DATA_PATH,
    chunksize=CHUNK_SIZE,
    low_memory=False,
    usecols=lambda c: c in FINAL_COLUMNS or c == 'source_file'
)

first_chunk = True

for chunk in chunks:

    # ✅ convert float64 → float32 (memory fix)
    fcols = chunk.select_dtypes('float').columns
    chunk[fcols] = chunk[fcols].astype('float32')

    cleaned_chunk = clean_deep_learning_data(chunk)

    mode = 'w' if first_chunk else 'a'
    header = first_chunk

    cleaned_chunk.to_csv(
        OUTPUT_PATH,
        index=False,
        mode=mode,
        header=header
    )

    first_chunk = False
    del cleaned_chunk

    print(f"✅ Processed chunk: {len(chunk)} rows")

print("🎉 Cleaning complete")

🚀 Starting cleaning...
✅ Processed chunk: 100000 rows
✅ Processed chunk: 100000 rows
✅ Processed chunk: 100000 rows
✅ Processed chunk: 100000 rows
✅ Processed chunk: 100000 rows
✅ Processed chunk: 100000 rows
✅ Processed chunk: 100000 rows
✅ Processed chunk: 100000 rows
✅ Processed chunk: 100000 rows
✅ Processed chunk: 100000 rows
✅ Processed chunk: 100000 rows
✅ Processed chunk: 100000 rows
✅ Processed chunk: 100000 rows
✅ Processed chunk: 100000 rows
✅ Processed chunk: 100000 rows
✅ Processed chunk: 100000 rows
✅ Processed chunk: 100000 rows
✅ Processed chunk: 100000 rows
✅ Processed chunk: 100000 rows
✅ Processed chunk: 100000 rows
✅ Processed chunk: 100000 rows
✅ Processed chunk: 100000 rows
✅ Processed chunk: 100000 rows
✅ Processed chunk: 100000 rows
✅ Processed chunk: 100000 rows
✅ Processed chunk: 100000 rows
✅ Processed chunk: 100000 rows
✅ Processed chunk: 100000 rows
✅ Processed chunk: 100000 rows
✅ Processed chunk: 100000 rows
✅ Processed chunk: 100000 rows
✅ Processed chun

In [15]:
df = pd.read_csv(OUTPUT_PATH)

print("✅ Final rows:", len(df))

✅ Final rows: 4844304
